# 🔥 Advanced · RAG — retrieval-augmented generation

> 🔥 **Advanced / heavy lab.** Ground an LLM's answers in your own documents: embed → index → retrieve → generate.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **10–15 min** including downloads. Built on the official **[sentence-transformers + FAISS](https://github.com/facebookresearch/faiss)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

RAG gives an LLM fresh, private, or citable knowledge without retraining it.

In [ ]:
!nvidia-smi -L
!pip install -q -U sentence-transformers faiss-cpu transformers accelerate

## 1 · A small corpus (swap in your own documents)

In [ ]:
docs = [
    "NeRF represents a scene as an MLP mapping (x,y,z,view) to colour and density, rendered by volume integration.",
    "3D Gaussian Splatting represents a scene as many anisotropic Gaussians, rasterized differentiably for real-time rendering.",
    "SMPL is a parametric human body model controlled by shape (beta) and pose (theta) parameters.",
    "SLAM jointly estimates the camera trajectory and a map of the environment from sensor data.",
    "A world model learns environment dynamics so an agent can plan by imagining future states.",
    "VideoMAE is a self-supervised video transformer pretrained by masking and reconstructing spatiotemporal patches.",
]

## 2 · Embed the corpus and build a FAISS index

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
emb = embedder.encode(docs, normalize_embeddings=True)
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
print("indexed", index.ntotal, "documents")

## 3 · Retrieve the most relevant passages

In [ ]:
def retrieve(q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True)
    scores, idx = index.search(qe, k)
    return [docs[i] for i in idx[0]]
print(retrieve("How are 3D scenes rendered in real time?"))

## 4 · Generate an answer grounded in the retrieved context

In [ ]:
from transformers import pipeline
llm = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", torch_dtype="auto", device_map="auto")
def answer(q):
    ctx = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(retrieve(q)))
    msgs = [{"role": "system", "content": "Answer using ONLY the context, and cite the [n] you used."},
            {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {q}"}]
    print(llm(msgs, max_new_tokens=200)[0]["generated_text"][-1]["content"])
answer("How does Gaussian Splatting render scenes, and how is it different from NeRF?")

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`.

## Next steps
- **Chunk** long docs, add a **reranker** (e.g. bge-reranker), and store vectors in **Chroma / Qdrant / pgvector**.
- Return **citations** with the answer; evaluate retrieval (recall@k) and answer faithfulness.
- Combine with a fine-tuned LLM (QLoRA lab) for a domain assistant.